<a href="https://colab.research.google.com/github/Harsh-Prajapati54/LLMs---Playbook/blob/main/Sementic_search_and_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Setup

In [9]:
%%capture
!pip install langchain==0.2.5 faiss-cpu==1.8.0 cohere==5.5.8 langchain-community==0.2.5 rank_bm25==0.2.2 sentence-transformers==3.0.1
!pip install llama-cpp-python==0.2.78  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

## IMPORTANT: Make sure to restart the session after installing the packages above.

In [1]:
# Step 1: Downgrade NumPy (run this first in a fresh cell)
!pip install -q "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 90.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cup

In [ ]:
# Run this in a cell FIRST, before any imports
!pip install -q --upgrade cohere numpy

# Then restart the runtime (required!)
import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 94.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.


In [ ]:
import os
os.kill(os.getpid(), 9)

# Dense Retrieval Example

### Getting the text archive and chunking it

In [1]:
import cohere
from google.colab import userdata

co = cohere.Client(userdata.get("cohere"))  # use `co`, not `cohere` directly

In [2]:
import cohere
import pandas as pd
import numpy as np
from tqdm import tqdm

In [3]:
texts = """ In 2000, Hamza Ali Mazari, then known as Jaskirat Singh Rangi, leaves his hometown of Pathankot to undergo military training. During his absence, a violent land dispute involving local MLA Sukhwinder Singh leads to brutal attacks on his family: his father is hanged and his sisters both gang raped, with his elder sister Harleen being murdered and younger sister Jasleen abducted. Upon returning, Jaskirat, with the help of his friend Gurbaaz, arms himself, murders Sukhwinder and the rest of his gang, and rescues Jasleen. He is subsequently arrested, tried, and sentenced to death. Before his incarceration, he entrusts Jasleen's future to Gurbaaz.[17]

In 2002, Jaskirat is abducted during a prison transfer and brought before intelligence officials Sushant Bansal and Ajay Sanyal. By 2004, he is recruited into a covert program to infiltrate Pakistani terror networks, given the new identity of Hamza Ali Mazari, and deployed to Kabul, where he severs ties with his past and operates under handler Mohammed Aalam.[c]

By 2009, following the death of Rehman Dakait in Lyari, Hamza manipulates local factions to ignite a war between the Baloch and Pathan gangs, resulting in the death of Pathan gang leader Arshad Pappu and the arrest of Rehman's brother Uzair Baloch, thereby consolidating Hamza's influence in Karachi while covertly advancing his mission. His actions draw him into the inner circle of "Bade Sahab" Dawood Ibrahim, who tasks him with facilitating narcotics operations to fund military operations. A group of Punjabi rebels are brought in to assist the operation, including Gurbaaz, shocking Hamza. During a party at Hamza's house, an intoxicated Gurbaaz confronts him in the bathroom and threatens to reveal his identity to the others, leading to a fight that ends with Gurbaaz dead. Aalam arrives to cover up the incident; when he is discovered, Hamza executes him in front of the guests.

As suspicion grows, SP Chaudhary Aslam begins investigating Hamza but is killed in a suicide attack orchestrated by Hamza through the Balochistan United Force (BUF). Yalina, who uncovers Hamza's true identity, confronts him but ultimately agrees to remain silent for the sake of their son, Zayan. Sanyal and Bansal grant Hamza full autonomy to aggressively dismantle the rest of the terror network, leading Hamza to execute several key operatives including financier Javed Khanani and IC 814 hijacker Zahoor Mistry. Meanwhile, Aslam's second-in-command Omar uncovers Hamza's true identity by forcing it out of Yalina, and informs Major Iqbal.

When Dawood plans a large-scale attack on India, Hamza travels to Iqbal's madrassa to deliver weapons; during their meeting, Iqbal confronts Hamza as an Indian spy and stabs him, but a bomb planted by Hamza and the BUF in the weapons crates goes off, killing multiple militants. A firefight ensues in the streets between Iqbal's Mujahideen and the BUF (disguised as Pakistan Rangers); Hamza destroys Iqbal's transport with an RPG, has the BUF level the madrassa, and chases Iqbal to a shipping yard, where they fight until Hamza kills Iqbal in a kerosene tanker explosion. Hamza is subsequently captured and tortured by Omar, until Sanyal coerces his release by blackmailing the head of Pakistani intelligence, Shamshad Hassan, with evidence of his dealings with Israeli intelligence. Uzair is used as a scapegoat and framed as having been India's spy all along.[19]

Hamza is dropped off at a rendezvous and picked up by his father-in-law Jameel Jamali, who reveals himself as a longtime Indian asset who had been slowly poisoning Dawood while embedded in his network. Having completed his mission, Hamza abandons his alias, forced to leave Yalina and their son behind, and returns to India as Jaskirat Singh Rangi. Though commended for his service, he disappears before his formal debriefing and returns to his childhood home, observing his family from a distance.[20]

In the mid-credits scene, flashbacks depict Jaskirat's training with the Research & Analysis Wing. In the post-credits scene, Shamshad orders Omar into a mental asylum after he threatens to reveal Shamshad's release of Hamza to the National Assembly. """

# split text into list of sentences
texts = texts.split('.')

# clean up to remove empty space and new line
texts = [t.strip('\n') for t in texts]

print(f"Total sentences: {len(texts)}")


Total sentences: 26


Lets now embedd the text chunks using the cohere api and get beck vector embedding model

In [4]:
# Get the embedding
response = co.embed(
    texts=texts,                    # must be a list
    model="embed-english-v3.0",      # specify model explicitly
    input_type="search_document",
).embeddings                         # plural

embeds = np.array(response)
print(embeds.shape)                  # expected: (1, 1024)

(26, 1024)


this means we have 26 vectors each one of embedding size 1024

### Building a Search Index

In [5]:
import faiss
dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
print(index.is_trained)
index.add(np.float32(embeds))

True


### search the index

In [6]:

def search(query, number_of_results=3):

    # 1. Get the query's embedding
    query_embed = co.embed(
        texts=[query],
        model="embed-english-v3.0",   # always specify model
        input_type="search_query",
    ).embeddings[0]

    # 2. Retrieve the nearest neighbors
    distances, similar_item_ids = index.search(np.float32([query_embed]), number_of_results)

    # 3. Format the results
    texts_np = np.array(texts)
    results = pd.DataFrame(data={
        'texts': texts_np[similar_item_ids[0]],
        'distance': distances[0]
    })

    # 4. Print and return the results
    print(f"Query: '{query}'\nNearest neighbors:")
    return results

In [7]:
query = "who was jaskirat singh Rangi"
results = search(query)
results

Query: 'who was jaskirat singh Rangi'
Nearest neighbors:


,texts,distance
0,"In 2000, Hamza Ali Mazari, then known as Jask...",0.934508
1,"Having completed his mission, Hamza abandons ...",0.957600
2,"[17]\n\nIn 2002, Jaskirat is abducted during a...",1.049600


# Reranking Example

In [8]:
# Step 1 — Get top 10 candidates from FAISS
results = search(query, number_of_results=10)

# Step 2 — Rerank to get best 3
reranked = co.rerank(
    query=query,
    documents=results['texts'].tolist(),
    top_n=3,
    model="rerank-english-v3.0",
)

for item in reranked.results:
    print(item.relevance_score, results['texts'].iloc[item.index])

Query: 'who was jaskirat singh Rangi'
Nearest neighbors:
0.992597  In 2000, Hamza Ali Mazari, then known as Jaskirat Singh Rangi, leaves his hometown of Pathankot to undergo military training
0.96653706  Having completed his mission, Hamza abandons his alias, forced to leave Yalina and their son behind, and returns to India as Jaskirat Singh Rangi
0.028761586 [17]

In 2002, Jaskirat is abducted during a prison transfer and brought before intelligence officials Sushant Bansal and Ajay Sanyal


In [9]:
query = "how precise was the science"
results = co.rerank(query=query, documents=texts, top_n=3, return_documents=True)
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text=' He is subsequently arrested, tried, and sentenced to death'), index=3, relevance_score=0.026559154),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text="When Dawood plans a large-scale attack on India, Hamza travels to Iqbal's madrassa to deliver weapons; during their meeting, Iqbal confronts Hamza as an Indian spy and stabs him, but a bomb planted by Hamza and the BUF in the weapons crates goes off, killing multiple militants"), index=16, relevance_score=0.023475897),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text=" A firefight ensues in the streets between Iqbal's Mujahideen and the BUF (disguised as Pakistan Rangers); Hamza destroys Iqbal's transport with an RPG, has the BUF level the madrassa, and chases Iqbal to a shipping yard, where they fight until Hamza kills Iqbal in a kerosene tanker explosion"), index=17, relevance_score=0.021078838)]